# Sparsity Thresholds LGBM

Compare several sparse-feature removal thresholds on the base processed feature set using fixed `LGBM` parameters.


In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, train_test_split

from src.loader import Loader
from src.modeling import build_lgbm_regressor
from src.preprocessing import FeaturePreprocessor

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
THRESHOLDS = [None, 0.975, 0.98, 0.9825, 0.985, 0.9875, 0.99]

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

Sparse filtering is applied only on the original features. Row-wise features are added after filtering so that each threshold is evaluated on its own final feature space.

In [8]:
best_params = {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 8,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.05,
    "reg_lambda": 0.05,
    "min_split_gain": 0.0,
}

best_params

{'n_estimators': 500,
 'learning_rate': 0.03,
 'num_leaves': 31,
 'max_depth': 8,
 'min_child_samples': 20,
 'subsample': 0.8,
 'subsample_freq': 1,
 'colsample_bytree': 0.8,
 'reg_alpha': 0.05,
 'reg_lambda': 0.05,
 'min_split_gain': 0.0}

In [10]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

In [ ]:
def evaluate_threshold(zero_share_threshold: float | None) -> dict:
    cv_scores = []
    cv_feature_counts = []

    for fold_train_idx, fold_valid_idx in cv.split(X_train_raw, y_train_log):
        X_fold_train_raw = X_train_raw.iloc[fold_train_idx]
        X_fold_valid_raw = X_train_raw.iloc[fold_valid_idx]
        y_fold_train_log = y_train_log.iloc[fold_train_idx]
        y_fold_valid_log = y_train_log.iloc[fold_valid_idx]

        preprocessor = FeaturePreprocessor(zero_share_threshold=zero_share_threshold)
        X_fold_train = preprocessor.fit_transform(X_fold_train_raw)
        X_fold_valid = preprocessor.transform(X_fold_valid_raw)

        cv_feature_counts.append(X_fold_train.shape[1])

        model = build_lgbm_regressor(best_params)
        model.fit(X_fold_train, y_fold_train_log)
        y_fold_pred_log = model.predict(X_fold_valid)
        fold_rmsle = root_mean_squared_error(y_fold_valid_log, y_fold_pred_log)
        cv_scores.append(fold_rmsle)

    preprocessor = FeaturePreprocessor(zero_share_threshold=zero_share_threshold)
    X_train_filtered = preprocessor.fit_transform(X_train_raw)
    X_test_filtered = preprocessor.transform(X_test_raw)

    model = build_lgbm_regressor(best_params)
    model.fit(X_train_filtered, y_train_log)
    y_pred_log = model.predict(X_test_filtered)
    y_pred = np.expm1(y_pred_log)
    y_pred = np.clip(y_pred, 0, None)

    return {
        "threshold": "none" if zero_share_threshold is None else zero_share_threshold,
        "features_kept": X_train_filtered.shape[1],
        "cv_features_mean": float(np.mean(cv_feature_counts)),
        "cv_rmsle_mean": float(np.mean(cv_scores)),
        "cv_rmsle_std": float(np.std(cv_scores)),
        "test_rmsle": float(root_mean_squared_log_error(y_test_raw, y_pred)),
        "test_rmse": float(root_mean_squared_error(y_test_raw, y_pred)),
        "test_mae": float(mean_absolute_error(y_test_raw, y_pred)),
        "test_r2": float(r2_score(y_test_raw, y_pred)),
    }

In [ ]:
results = [evaluate_threshold(threshold) for threshold in THRESHOLDS]
results_df = pd.DataFrame(results).sort_values(by="cv_rmsle_mean").reset_index(drop=True)
results_df.style.format({
    "cv_rmsle_mean": "{:,.4f}",
    "cv_rmsle_std": "{:,.4f}",
    "test_rmsle": "{:,.4f}",
    "test_rmse": "{:,.0f}",
    "test_mae": "{:,.0f}",
    "test_r2": "{:,.4f}",
    "cv_features_mean": "{:,.1f}",
})

threshold  features_kept  cv_features_mean  cv_rmsle_mean  cv_rmsle_std  test_rmsle    test_rmse     test_mae  test_r2
   0.9875           2482            2498.2       1.449950      0.039943    1.463481 7.438939e+06 4.209312e+06 0.132963
     none           4731            4731.0       1.453664      0.044582    1.467131 7.434416e+06 4.172218e+06 0.134016
     0.99           2669            2682.8       1.456767      0.042673    1.459197 7.384670e+06 4.145083e+06 0.145567
    0.975           1832            1835.2       1.457617      0.041252    1.457825 7.381933e+06 4.162411e+06 0.146200
   0.9825           2287            2293.2       1.460002      0.045543    1.466030 7.417042e+06 4.180470e+06 0.138059
    0.985           2415            2407.8       1.460065      0.042820    1.461717 7.408897e+06 4.185611e+06 0.139951
     0.98           2113            2125.6       1.462663      0.034740    1.448877 7.371678e+06 4.143438e+06 0.148571

In [ ]:
summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "thresholds": ["none" if threshold is None else threshold for threshold in THRESHOLDS],
    "model_params": best_params,
    "results": results_df.to_dict(orient="records"),
}

summary

{'target_transform': 'log1p', 'primary_metric': 'rmsle', 'thresholds': ['none', 0.975, 0.98, 0.9825, 0.985, 0.9875, 0.99], 'model_params': {'n_estimators': 500, 'learning_rate': 0.03, 'num_leaves': 31, 'max_depth': 8, 'min_child_samples': 20, 'subsample': 0.8, 'subsample_freq': 1, 'colsample_bytree': 0.8, 'reg_alpha': 0.05, 'reg_lambda': 0.05, 'min_split_gain': 0.0}, 'results': [{'threshold': 0.9875, 'features_kept': 2482, 'cv_features_mean': 2498.2, 'cv_rmsle_mean': 1.4499500171495303, 'cv_rmsle_std': 0.03994323426878241, 'test_rmsle': 1.463480617652269, 'test_rmse': 7438938.856782915, 'test_mae': 4209311.635438668, 'test_r2': 0.13296252756429394}, {'threshold': 'none', 'features_kept': 4731, 'cv_features_mean': 4731.0, 'cv_rmsle_mean': 1.4536642109426965, 'cv_rmsle_std': 0.04458214103952521, 'test_rmsle': 1.4671306138157396, 'test_rmse': 7434416.204750145, 'test_mae': 4172218.104378253, 'test_r2': 0.1340164727152141}, {'threshold': 0.99, 'features_kept': 2669, 'cv_features_mean': 268

## How To Read The Results

- Read the ranking by `cv_rmsle_mean` first.
- Accept a threshold only if the test `RMSLE` does not regress materially.
- If one threshold wins clearly, rerun `Optuna` on that filtered feature space rather than trying many more sparse cutoffs.

# 05 Sparsity Thresholds LGBM Report

## Goal

The goal of this notebook was to find a good sparse feature threshold using only the base processed feature set.

## What Was Done

- Loaded `data/processed_data.csv`.
- Split the data into train and test parts.
- Tested zero-share thresholds: `none`, `0.975`, `0.98`, `0.9825`, `0.985`, `0.9875`, and `0.99`.
- For each CV fold, selected sparse columns using only the fold train data.
- Trained LightGBM on the filtered base features without row-wise aggregate features.
- Compared thresholds by CV RMSLE in log-target space.
- Evaluated each threshold on the held-out test split.

## Threshold Results

| threshold | features kept | mean CV features | CV RMSLE mean | CV RMSLE std | test RMSLE | test RMSE | test MAE | test R2 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `0.9875` | 2,482 | 2,498.2 | 1.4500 | 0.0399 | 1.4635 | 7,438,939 | 4,209,312 | 0.1330 |
| `none` | 4,731 | 4,731.0 | 1.4537 | 0.0446 | 1.4671 | 7,434,416 | 4,172,218 | 0.1340 |
| `0.99` | 2,669 | 2,682.8 | 1.4568 | 0.0427 | 1.4592 | 7,384,670 | 4,145,083 | 0.1456 |
| `0.975` | 1,832 | 1,835.2 | 1.4576 | 0.0413 | 1.4578 | 7,381,933 | 4,162,411 | 0.1462 |
| `0.9825` | 2,287 | 2,293.2 | 1.4600 | 0.0455 | 1.4660 | 7,417,042 | 4,180,470 | 0.1381 |
| `0.985` | 2,415 | 2,407.8 | 1.4601 | 0.0428 | 1.4617 | 7,408,897 | 4,185,611 | 0.1400 |
| `0.98` | 2,113 | 2,125.6 | 1.4627 | 0.0347 | 1.4489 | 7,371,678 | 4,143,438 | 0.1486 |

## Main Result

The best threshold by CV RMSLE was `0.9875`.

| metric | value |
| --- | ---: |
| Best CV threshold | 0.9875 |
| Features kept | 2,482 |
| Mean CV features | 2,498.2 |
| CV RMSLE mean | 1.4500 |
| CV RMSLE std | 0.0399 |
| Test RMSLE | 1.4635 |
| Test RMSE | 7,438,939 |
| Test MAE | 4,209,312 |
| Test R2 | 0.1330 |

The best held-out test RMSLE in the output was from threshold `0.98` with test RMSLE `1.4489`, but this notebook selects the sparse threshold by CV RMSLE.

## Conclusion

Sparse feature filtering alone gives only a small CV improvement over using all base features (`1.4500` vs `1.4537`). The CV-selected threshold is `0.9875`, keeping `2,482` base features.

This notebook intentionally does not add row-wise aggregate features; those are tested separately in notebook `06`. The best held-out test RMSLE in this run is from threshold `0.98`, but threshold selection is based on CV RMSLE, so `0.9875` is the CV-selected result.
